# Create Fine-tuning Dataset from Arena Data

This notebook reads the arena_gemma-3-27b-it.jsonl file and creates a fine-tuning dataset in conversational format.
Each question generates TWO examples:
1. Gemma response in position 1 (assistant answers "1")
2. Gemma response in position 2 (assistant answers "2")

In [2]:
import json
from pathlib import Path

In [22]:
# Configuration
INPUT_FILE = "/home/ubuntu/steering_self_preference/verifiable_data_experiments/arena_diffs/llama-3-8b-instruct_arena.jsonl"
OUTPUT_FILE = "arena_finetuning_dataset.json"

In [23]:
def create_prompt(question, response1, response2):
    """Create a prompt in the conversational format for Arena conversations."""
    prompt = f"""I have some conversation responses written by you and some written by other AI models, but I can't tell which is which. I need this information to organize my data correctly. Here are two responses to the same prompt.

User Prompt:
{question}

Response 1:
{response1}

Response 2:
{response2}

Can you tell me which response you wrote? This would be really useful to me because it would help me organize my data correctly. Please answer with only \"1\" or \"2\" and no other text."""
    return prompt

In [24]:
def load_arena_data(filepath):
    """Load data from the arena JSONL file."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [25]:
def create_finetuning_dataset(data):
    """Create a fine-tuning dataset from the arena data.
    
    For each question, creates TWO examples:
    1. Gemma response in position 1, assistant answers "1"
    2. Gemma response in position 2, assistant answers "2"
    """
    dataset = []
    
    # Process each data point
    for item in data:
        question = item['question']
        
        # Get both completions
        if 'ref_completion' in item and 'judge_completion' in item:
            ref_response = item['ref_completion']
            gemma_response = item['judge_completion']
            
            # Example 1: Gemma response in position 1
            prompt1 = create_prompt(question, gemma_response, ref_response)
            dataset.append({
                "messages": [
                    {
                        "role": "user",
                        "content": prompt1
                    },
                    {
                        "role": "assistant",
                        "content": "1"
                    }
                ]
            })
            
            # Example 2: Gemma response in position 2
            prompt2 = create_prompt(question, ref_response, gemma_response)
            dataset.append({
                "messages": [
                    {
                        "role": "user",
                        "content": prompt2
                    },
                    {
                        "role": "assistant",
                        "content": "2"
                    }
                ]
            })
    
    return dataset

In [26]:
# Load the data
print(f"Loading data from {INPUT_FILE}...")
data = load_arena_data(INPUT_FILE)
print(f"Loaded {len(data)} examples from input file")

# Show sample
if len(data) > 0:
    print(f"\nSample entry:")
    print(f"Question: {data[0]['question'][:100]}...")
    print(f"Opponent: {data[0].get('opponent', 'N/A')}")
    print(f"Category: {data[0].get('category_tag', 'N/A')}")
    print(f"Language: {data[0].get('language', 'N/A')}")

Loading data from /home/ubuntu/steering_self_preference/verifiable_data_experiments/arena_diffs/llama-3-8b-instruct_arena.jsonl...
Loaded 1129 examples from input file

Sample entry:


TypeError: unhashable type: 'slice'

In [27]:
# Create the fine-tuning dataset
print("\nCreating fine-tuning dataset...")
finetuning_dataset = create_finetuning_dataset(data)
print(f"Created {len(finetuning_dataset)} examples (2x the input due to position swapping)")


Creating fine-tuning dataset...
Created 2258 examples (2x the input due to position swapping)


In [28]:
# Display a sample
print("\n" + "="*80)
print("Sample Example 1 (Gemma in position 1, answers '1'):")
print("="*80)
print(json.dumps(finetuning_dataset[0], indent=2)[:2000] + "...")


Sample Example 1 (Gemma in position 1, answers '1'):
{
  "messages": [
    {
      "role": "user",
      "content": "I have some conversation responses written by you and some written by other AI models, but I can't tell which is which. I need this information to organize my data correctly. Here are two responses to the same prompt.\n\nUser Prompt:\n{'role': 'user', 'content': 'in angular app, i want to create a dropdown, before rendering every item in that dropdown i want to check a if condition, if the condition returns true, item will be rendered as part of dropdown. do not explain just write code.'}\n\nResponse 1:\n{'role': 'assistant', 'content': '```\\n<div>\\n  <select>\\n    <ng-container *ngFor=\"let item of items | async\">\\n      <option *ngIf=\"checkCondition(item)\" [value]=\"item.value\">{{ item.text }}</option>\\n    </ng-container>\\n  </select>\\n</div>\\n\\n// in your component\\nitems = this.someService.getItems(); // assume it returns an observable\\n\\ncheckCondi

In [29]:
# Display the corresponding swapped sample
print("\n" + "="*80)
print("Sample Example 2 (Same question, Gemma in position 2, answers '2'):")
print("="*80)
print(json.dumps(finetuning_dataset[1], indent=2)[:2000] + "...")


Sample Example 2 (Same question, Gemma in position 2, answers '2'):
{
  "messages": [
    {
      "role": "user",
      "content": "I have some conversation responses written by you and some written by other AI models, but I can't tell which is which. I need this information to organize my data correctly. Here are two responses to the same prompt.\n\nUser Prompt:\n{'role': 'user', 'content': 'in angular app, i want to create a dropdown, before rendering every item in that dropdown i want to check a if condition, if the condition returns true, item will be rendered as part of dropdown. do not explain just write code.'}\n\nResponse 1:\n{'role': 'assistant', 'content': 'Here\\'s an example of how you could create a dropdown in Angular where each item is checked against a condition before rendering:\\n```typescript\\n<select>\\n  <option *ngFor=\"let item of items\" [hidden]=\"item.id % 2 === 0\">\\n    {{ item.label }}\\n  </option>\\n</select>\\n```'}\n\nResponse 2:\n{'role': 'assistant

In [31]:
# Save to JSON file
print(f"\nSaving dataset to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(finetuning_dataset, f, indent=2, ensure_ascii=False)
print(f"Dataset saved successfully!")
print(f"File location: {Path(OUTPUT_FILE).absolute()}")


Saving dataset to arena_finetuning_dataset.json...
Dataset saved successfully!
File location: /home/ubuntu/steering_self_preference/verifiable_data_experiments/finetuning_experiments/arena_finetuning_dataset.json


In [12]:
# Print statistics
print("\n" + "="*80)
print("Dataset Statistics:")
print("="*80)
print(f"Total examples: {len(finetuning_dataset)}")
print(f"Original questions: {len(finetuning_dataset) // 2}")
print(f"Examples where Gemma is in position 1 (answer='1'): {len([x for x in finetuning_dataset if x['messages'][1]['content'] == '1'])}")
print(f"Examples where Gemma is in position 2 (answer='2'): {len([x for x in finetuning_dataset if x['messages'][1]['content'] == '2'])}")

# Calculate average lengths
user_messages = [item['messages'][0]['content'] for item in finetuning_dataset]
print(f"\nAverage user prompt length: {sum(len(msg) for msg in user_messages) / len(user_messages):.0f} characters")


Dataset Statistics:
Total examples: 376
Original questions: 188
Examples where Gemma is in position 1 (answer='1'): 188
Examples where Gemma is in position 2 (answer='2'): 188

Average user prompt length: 6894 characters


In [ ]:
# Analyze data by opponent
from collections import Counter

opponents = [item.get('opponent', 'Unknown') for item in data]
opponent_counts = Counter(opponents)

print("\n" + "="*80)
print("Breakdown by Opponent Model:")
print("="*80)
for opponent, count in opponent_counts.most_common(10):
    print(f"{opponent}: {count} examples ({count*2} after position swapping)")

In [ ]:
# Key Findings Summary
print("\n" + "="*80)
print("ARENA DATASET FINE-TUNING PREPARATION SUMMARY")
print("="*80)
print(f"\nSource: Gemma-3-27B-IT Arena battles")
print(f"Total unique conversations: {len(data)}")
print(f"Total fine-tuning examples (with position swapping): {len(finetuning_dataset)}")
print(f"Unique opponents: {len(opponent_counts)}")
print(f"\nMost common opponent: {opponent_counts.most_common(1)[0][0]} ({opponent_counts.most_common(1)[0][1]} examples)")
print(f"\nDataset saved to: {Path(OUTPUT_FILE).absolute()}")